In [ ]:
import os
import shutil
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from scipy.interpolate import interp1d
from scipy.stats import norm

warnings.simplefilter(action="ignore", category=FutureWarning)

In [ ]:
onder_161 = 1 / 30000
sig_161 = 1 / 100000
onder_162 = 1 / 10000
sig_162 = 1 / 30000

## Kerende hoogte toets

In [ ]:
loc_data = r"P:\PR\5536.10\Uitvoering\04 - Fase 1\04 - Analyse\MOEDERTABEL - Informatie per vak.xlsx"
loc_template = (
    r"P:\PR\5536.10\Uitvoering\04 - Fase 1\04 - Analyse\03_Logboeken\templates\template_faalpad_geometrische_toets.xlsx"
)
loc_logboek = r"P:\PR\5536.10\Uitvoering\04 - Fase 1\04 - Analyse\03_Logboeken"
data = pd.read_excel(loc_data)

In [ ]:
min_waterstand = 1
max_waterstand = float(data["Hoogte kruin"].dropna().max())
delta_h = 0.001  # 0.001
data.head()
data["vakkans"] = np.nan

### 16-1

In [ ]:
for i in [54]:  # vervang 1 door len(data) voor alle locaties
    print(i)
    if str(data["Beoordelen?"][i]).strip().lower() != "ja":
        continue

    locatie_naam = data["HR locatie"][i]
    print(f"Verwerken: {locatie_naam}")

    # Waterstand-grid
    voorland = data["Hoogte voorland"][i] + data["Bodemdaling totaal in 2035 [m] "][i]
    achterland = data["Hoogte achterland"][i] + data["Bodemdaling totaal in 2035 [m] "][i]
    kruin = data["Hoogte kruin"][i] + data["Bodemdaling totaal in 2035 [m] "][i]
    h_grid = np.arange(max(min(voorland, achterland) - 1, min_waterstand), min(kruin + 1, max_waterstand), delta_h)
    h_overstroming = data.loc[i, "Waterstand waar overstroming op treedt"]
    uitintegreren = pd.DataFrame(
        {
            "Waterstand": h_grid,
            "CDF": np.nan,
            "Kans op waterstand, PDF": np.nan,
            "Conditionele faalkans": 0.0,
            "Faalkansbijdrage": np.nan,
            "Beta": np.nan,
        }
    )

    # Pad naar hfreq.txt
    hfreq_path = os.path.join(loc_161, locatie_naam, "Berekeningen", "ws", "hfreq.txt")
    if not os.path.exists(hfreq_path):
        print(f"Geen hfreq gevonden: {hfreq_path}")
        continue

    # Lees hfreq: waterstanden + waarden (hier geïnterpreteerd als λ [1/jaar])
    water, waarden = read_hfreq(hfreq_path)
    # Interpoleer/extrapoleer λ op het grid
    f_lam = interp1d(water, np.log10(waarden), kind="linear", fill_value="extrapolate", bounds_error=False)
    lam = 10 ** f_lam(h_grid)

    # λ kan niet negatief; clip naar [0, inf)
    lam = np.clip(lam, 0.0, np.inf)

    # CDF uit λ (1-jaars): F(h)=exp(-λ)
    uitintegreren["CDF"] = np.exp(-lam)

    # PDF uit CDF: f(h_i) ≈ (F_i - F_{i-1}) / Δh
    pdf = (uitintegreren["CDF"] - uitintegreren["CDF"].shift(1)) / delta_h
    pdf.iloc[0] = 0.0
    pdf = pdf.clip(lower=0.0)  # numerieke ruis opvangen
    uitintegreren["Kans op waterstand, PDF"] = pdf

    # Fragility curve (stapfunctie): 1 tussen h_onder en kruin, anders 0

    h_onder = max(voorland, achterland, h_overstroming)
    dh = kruin - achterland
    h_karakteristiek = np.array(
        [achterland, achterland + 0.25 * dh, achterland + 0.5 * dh, achterland + 0.75 * dh, kruin]
    )
    beta_schatten = np.array([10, 6, 4, 1.5, 1])
    uitintegreren["Beta"] = np.interp(uitintegreren["Waterstand"], h_karakteristiek, beta_schatten)
    uitintegreren["Conditionele faalkans"] = norm.cdf(-uitintegreren["Beta"])

    # logboek maken
    nieuwe_naam = f"{data.Traject[i]}_{data.vaknummer[i]}_{data.Vaknaam[i]}.xlsx"
    doel_pad = os.path.join(loc_logboek, nieuwe_naam)
    metadata = pd.DataFrame(
        {
            "TRAJECT_ID": [data.Traject[i]],
            "M_VAN": [data.M_van[i]],
            "M_TOT": [data.M_tot[i]],
            "dijkvaknummer": [data.vaknummer[i]],
            "Vaknaam": [data.Vaknaam[i]],
            "LENGTE_VAK": [data.Lengte_vak[i]],
            "TYPE_WATERKERING": [data.Eigenschap[i]],
            "Template": [data.TYPE_WATERKERING[i]],
            "Ondergrondscenario": [1],
            "ScenarioKans": [1],
            "M_profiel": [data.M_profiel[i]],
            "dwp_naam": [data.M_profiel[i]],
            "HR_locatie": [data["HR locatie"][i]],
            "kruinhoogte": [data["Hoogte kruin"][i]],
        }
    )
    shutil.copy2(loc_template, doel_pad)
    wb = load_workbook(doel_pad)
    ws = wb.active  # of wb["NaamVanSheet"]

    start_row = 6
    start_col = 1  # A

    if metadata.shape != (1, 14):
        raise ValueError("metadata moet exact 1×13 zijn")

    for col_offset in range(14):
        ws.cell(row=start_row, column=start_col + col_offset).value = metadata.iloc[0, col_offset]
    start_row = 27
    kans_gebruik = norm.cdf(-beta_schatten)
    if h_onder < achterland:
        # print (0)
        waarden_D = np.insert(h_karakteristiek, 0, h_karakteristiek[0] - 0.0001)
        waarden_E = np.insert(kans_gebruik, 0, 0)
    elif h_onder < achterland + 0.25 * dh:
        # print (1)
        betanew = np.interp(h_onder, h_karakteristiek[0:2], beta_schatten[0:2])
        kans_gebruik[h_karakteristiek < h_onder] = 0
        waarden_D = np.insert(h_karakteristiek, 1, [h_onder - 0.0001, h_onder])
        waarden_E = np.insert(kans_gebruik, 1, [0, norm.cdf(-betanew)])
    elif h_onder < achterland + 0.5 * dh:
        # print (2)
        betanew = np.interp(h_onder, h_karakteristiek[1:3], beta_schatten[1:3])
        kans_gebruik[h_karakteristiek < h_onder] = 0
        waarden_D = np.insert(h_karakteristiek, 2, [h_onder - 0.0001, h_onder])
        waarden_E = np.insert(kans_gebruik, 2, [0, norm.cdf(-betanew)])
    elif h_onder < achterland + 0.75 * dh:
        # print (3)
        betanew = np.interp(h_onder, h_karakteristiek[2:4], beta_schatten[2:4])
        kans_gebruik[h_karakteristiek < h_onder] = 0
        waarden_D = np.insert(h_karakteristiek, 3, [h_onder - 0.0001, h_onder])
        waarden_E = np.insert(kans_gebruik, 3, [0, norm.cdf(-betanew)])
    elif h_onder < kruin:
        # print (4)
        betanew = np.interp(h_onder, h_karakteristiek[3:5], beta_schatten[3:5])
        kans_gebruik[h_karakteristiek < h_onder] = 0
        waarden_D = np.insert(h_karakteristiek, 4, [h_onder - 0.0001, h_onder])
        waarden_E = np.insert(kans_gebruik, 4, [0, norm.cdf(-betanew)])
    elif h_onder > kruin:
        waarden_D = 0
        waarden_E = 1
    # plt.plot(waarden_D,waarden_E,marker='o')
    # print (h_onder, achterland, kruin, betanew)
    kolom_data = {1: np.ones(len(waarden_D)), 2: np.ones(len(waarden_D)), 4: waarden_D, 5: waarden_E}
    for col, waarden in kolom_data.items():
        for j, waarde in enumerate(waarden):
            ws.cell(row=start_row + j, column=col).value = waarde
    wb.save(doel_pad)
    wb.close()

    mask = uitintegreren["Waterstand"] < h_onder
    uitintegreren.loc[mask, "Conditionele faalkans"] = 0.0

    # Kansbijdrage
    uitintegreren["Faalkansbijdrage"] = (
        uitintegreren["Conditionele faalkans"] * uitintegreren["Kans op waterstand, PDF"]
    )
    uitintegreren["Faalkansbijdrage"][uitintegreren["Waterstand"] > kruin] = 0

    # Faalkans = integraal van bijdrage over h: Pf ≈ Σ (bijdrage * Δh)
    Faalkans = (uitintegreren["Faalkansbijdrage"] * delta_h).sum()
    fig, ax1 = plt.subplots(figsize=(10, 6))

    # =========================
    # Linker y-as (kansen)
    # =========================
    (l1,) = ax1.plot(
        uitintegreren["Waterstand"], uitintegreren["CDF"], label="CDF  F(h)=P(H≤h)", color="tab:blue", linewidth=2
    )

    (l2,) = ax1.step(
        uitintegreren["Waterstand"],
        uitintegreren["Conditionele faalkans"],
        where="post",
        label="Fragility  P(falen|h)",
        color="tab:red",
        linewidth=2,
    )

    ax1.set_xlabel("Waterstand h [m]")
    ax1.set_ylabel("Kans [-]")
    ax1.set_ylim(-0.05, 1.05)
    ax1.grid(True, alpha=0.3)

    # =========================
    # Rechter y-as (dichtheden)
    # =========================
    ax2 = ax1.twinx()
    (l3,) = ax2.plot(
        uitintegreren["Waterstand"],
        uitintegreren["Kans op waterstand, PDF"],
        label="PDF  f_H(h)",
        color="tab:orange",
        linestyle="--",
    )

    # =========================
    # Kansbijdrage als histogram
    # =========================
    bars = ax2.bar(
        uitintegreren["Waterstand"],
        uitintegreren["Faalkansbijdrage"] * delta_h,
        width=delta_h,
        label="Kansbijdrage per Δh",
        color="tab:green",
        alpha=0.6,
        align="center",
    )

    ax2.set_ylabel("Kans per waterstandsinterval [-]")

    # -------------------------
    # Verticale lijnen + labels
    # -------------------------
    for x, lab in [
        (voorland, "Hoogte voorland"),
        (achterland, "Hoogte achterland"),
        (kruin, "Kruinhoogte"),
    ]:
        ax1.axvline(x=x, color="k", linestyle=":", linewidth=1.5, alpha=0.2)
        ax1.text(
            x,
            0.1,
            lab,
            transform=ax1.get_xaxis_transform(),
            rotation=90,
            va="baseline",
            ha="right",
            fontsize=9,
            alpha=0.2,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.7, pad=1.5),
        )

    # =========================
    # Legenda samenvoegen
    # =========================
    lines = [l1, l2, l3, bars]
    labels = ["CDF  F(h)=P(H≤h)", "Fragility  P(falen|h)", "PDF  f_H(h)", "Kansbijdrage per Δh"]

    ax1.legend(lines, labels, loc="upper right", frameon=True)

    # =========================
    # Titel
    # =========================
    traj = str(data.Traject[i])
    vak = str(data.vaknummer[i])
    ax1.set_title(f"Vaknummer {vak}\nKerende hoogte dh = {dh:.2f} m\nTotale faalkans Pf = {Faalkans:.2e}")
    ax2.set_yscale("log")
    plt.tight_layout()
    plt.show()

    print(f"De faalkans bedraagt Pf = {Faalkans:.6e}")
    data.iloc[i, -1] = Faalkans
    out_base = r"P:\PR\5536.10\Uitvoering\04 - Fase 1\04 - Analyse\02_geometrischetoets"
    os.makedirs(out_base, exist_ok=True)

    fig_path = os.path.join(out_base, f"{traj}_{vak}.png")

    plt.tight_layout()
    fig.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    plt.close()
    print(i)

### 16-2

In [ ]:
for i in range(68, len(data)):  # vervang 1 door len(data) voor alle locaties
    print(i)
    if str(data["Beoordelen?"][i]).strip().lower() != "ja":
        continue

    locatie_naam = data["HR locatie"][i]
    print(f"Verwerken: {locatie_naam}")

    # Waterstand-grid
    voorland = data["Hoogte voorland"][i] + data["Bodemdaling totaal in 2035 [m] "][i]
    achterland = data["Hoogte achterland"][i] + data["Bodemdaling totaal in 2035 [m] "][i]
    kruin = data["Hoogte kruin"][i] + data["Bodemdaling totaal in 2035 [m] "][i]
    h_grid = np.arange(max(min(voorland, achterland) - 1, min_waterstand), min(kruin + 1, max_waterstand), delta_h)
    h_overstroming = data.loc[i, "Waterstand waar overstroming op treedt"]
    uitintegreren = pd.DataFrame(
        {
            "Waterstand": h_grid,
            "CDF": np.nan,
            "Kans op waterstand, PDF": np.nan,
            "Conditionele faalkans": 0.0,
            "Faalkansbijdrage": np.nan,
            "Beta": np.nan,
        }
    )

    # Pad naar hfreq.txt
    hfreq_path = os.path.join(loc_162, locatie_naam, "Berekeningen", "ws", "hfreq.txt")
    if not os.path.exists(hfreq_path):
        print(f"Geen hfreq gevonden: {hfreq_path}")
        continue

    # Lees hfreq: waterstanden + waarden (hier geïnterpreteerd als λ [1/jaar])
    water, waarden = read_hfreq(hfreq_path)
    # Interpoleer/extrapoleer λ op het grid
    f_lam = interp1d(water, np.log10(waarden), kind="linear", fill_value="extrapolate", bounds_error=False)
    lam = 10 ** f_lam(h_grid)

    # λ kan niet negatief; clip naar [0, inf)
    lam = np.clip(lam, 0.0, np.inf)

    # CDF uit λ (1-jaars): F(h)=exp(-λ)
    uitintegreren["CDF"] = np.exp(-lam)

    # PDF uit CDF: f(h_i) ≈ (F_i - F_{i-1}) / Δh
    pdf = (uitintegreren["CDF"] - uitintegreren["CDF"].shift(1)) / delta_h
    pdf.iloc[0] = 0.0
    pdf = pdf.clip(lower=0.0)  # numerieke ruis opvangen
    uitintegreren["Kans op waterstand, PDF"] = pdf

    # Fragility curve (stapfunctie): 1 tussen h_onder en kruin, anders 0

    h_onder = max(voorland, achterland, h_overstroming)
    dh = kruin - achterland
    h_karakteristiek = np.array(
        [achterland, achterland + 0.25 * dh, achterland + 0.5 * dh, achterland + 0.75 * dh, kruin]
    )
    beta_schatten = np.array([10, 6, 4, 1.5, 1])
    uitintegreren["Beta"] = np.interp(uitintegreren["Waterstand"], h_karakteristiek, beta_schatten)
    uitintegreren["Conditionele faalkans"] = norm.cdf(-uitintegreren["Beta"])

    # logboek maken
    nieuwe_naam = f"{data.Traject[i]}_{data.vaknummer[i]}_{data.Vaknaam[i]}.xlsx"
    doel_pad = os.path.join(loc_logboek, nieuwe_naam)
    metadata = pd.DataFrame(
        {
            "TRAJECT_ID": [data.Traject[i]],
            "M_VAN": [data.M_van[i]],
            "M_TOT": [data.M_tot[i]],
            "dijkvaknummer": [data.vaknummer[i]],
            "Vaknaam": [data.Vaknaam[i]],
            "LENGTE_VAK": [data.Lengte_vak[i]],
            "TYPE_WATERKERING": [data.Eigenschap[i]],
            "Template": [data.TYPE_WATERKERING[i]],
            "Ondergrondscenario": [1],
            "ScenarioKans": [1],
            "M_profiel": [data.M_profiel[i]],
            "dwp_naam": [data.M_profiel[i]],
            "HR_locatie": [data["HR locatie"][i]],
            "kruinhoogte": [data["Hoogte kruin"][i]],
        }
    )
    shutil.copy2(loc_template, doel_pad)
    wb = load_workbook(doel_pad)
    ws = wb.active  # of wb["NaamVanSheet"]

    start_row = 6
    start_col = 1  # A

    if metadata.shape != (1, 14):
        raise ValueError("metadata moet exact 1×13 zijn")

    for col_offset in range(14):
        ws.cell(row=start_row, column=start_col + col_offset).value = metadata.iloc[0, col_offset]
    start_row = 27
    kans_gebruik = norm.cdf(-beta_schatten)
    if h_onder < achterland:
        print(0)
        waarden_D = np.insert(h_karakteristiek, 0, h_karakteristiek[0] - 0.0001)
        waarden_E = np.insert(kans_gebruik, 0, 0)
    elif h_onder < achterland + 0.25 * dh:
        print(1)
        betanew = np.interp(h_onder, h_karakteristiek[0:2], beta_schatten[0:2])
        kans_gebruik[h_karakteristiek < h_onder] = 0
        waarden_D = np.insert(h_karakteristiek, 1, [h_onder - 0.0001, h_onder])
        waarden_E = np.insert(kans_gebruik, 1, [0, norm.cdf(-betanew)])
    elif h_onder < achterland + 0.5 * dh:
        print(2)
        betanew = np.interp(h_onder, h_karakteristiek[1:3], beta_schatten[1:3])
        kans_gebruik[h_karakteristiek < h_onder] = 0
        waarden_D = np.insert(h_karakteristiek, 2, [h_onder - 0.0001, h_onder])
        waarden_E = np.insert(kans_gebruik, 2, [0, norm.cdf(-betanew)])
    elif h_onder < achterland + 0.75 * dh:
        print(3)
        betanew = np.interp(h_onder, h_karakteristiek[2:4], beta_schatten[2:4])
        kans_gebruik[h_karakteristiek < h_onder] = 0
        waarden_D = np.insert(h_karakteristiek, 3, [h_onder - 0.0001, h_onder])
        waarden_E = np.insert(kans_gebruik, 3, [0, norm.cdf(-betanew)])
    elif h_onder < kruin:
        print(4)
        betanew = np.interp(h_onder, h_karakteristiek[3:5], beta_schatten[3:5])
        kans_gebruik[h_karakteristiek < h_onder] = 0
        waarden_D = np.insert(h_karakteristiek, 4, [h_onder - 0.0001, h_onder])
        waarden_E = np.insert(kans_gebruik, 4, [0, norm.cdf(-betanew)])
    elif h_onder >= kruin:
        print(5)
        print(h_onder, kruin)
        waarden_D = [min_waterstand, kruin]
        waarden_E = np.zeros(len(waarden_D))
    # plt.plot(waarden_D,waarden_E,marker='o')
    print(h_onder, achterland, kruin, betanew)
    kolom_data = {1: np.ones(len(waarden_D)), 2: np.ones(len(waarden_D)), 4: waarden_D, 5: waarden_E}
    for col, waarden in kolom_data.items():
        for j, waarde in enumerate(waarden):
            ws.cell(row=start_row + j, column=col).value = waarde
    wb.save(doel_pad)
    wb.close()

    mask = uitintegreren["Waterstand"] < h_onder
    uitintegreren.loc[mask, "Conditionele faalkans"] = 0.0

    # Kansbijdrage
    uitintegreren["Faalkansbijdrage"] = (
        uitintegreren["Conditionele faalkans"] * uitintegreren["Kans op waterstand, PDF"]
    )
    uitintegreren["Faalkansbijdrage"][uitintegreren["Waterstand"] > kruin] = 0

    # Faalkans = integraal van bijdrage over h: Pf ≈ Σ (bijdrage * Δh)
    Faalkans = (uitintegreren["Faalkansbijdrage"] * delta_h).sum()
    fig, ax1 = plt.subplots(figsize=(10, 6))

    # =========================
    # Linker y-as (kansen)
    # =========================
    (l1,) = ax1.plot(
        uitintegreren["Waterstand"], uitintegreren["CDF"], label="CDF  F(h)=P(H≤h)", color="tab:blue", linewidth=2
    )

    (l2,) = ax1.step(
        uitintegreren["Waterstand"],
        uitintegreren["Conditionele faalkans"],
        where="post",
        label="Fragility  P(falen|h)",
        color="tab:red",
        linewidth=2,
    )

    ax1.set_xlabel("Waterstand h [m]")
    ax1.set_ylabel("Kans [-]")
    ax1.set_ylim(-0.05, 1.05)
    ax1.grid(True, alpha=0.3)

    # =========================
    # Rechter y-as (dichtheden)
    # =========================
    ax2 = ax1.twinx()
    (l3,) = ax2.plot(
        uitintegreren["Waterstand"],
        uitintegreren["Kans op waterstand, PDF"],
        label="PDF  f_H(h)",
        color="tab:orange",
        linestyle="--",
    )

    # =========================
    # Kansbijdrage als histogram
    # =========================
    bars = ax2.bar(
        uitintegreren["Waterstand"],
        uitintegreren["Faalkansbijdrage"] * delta_h,
        width=delta_h,
        label="Kansbijdrage per Δh",
        color="tab:green",
        alpha=0.6,
        align="center",
    )

    ax2.set_ylabel("Kans per waterstandsinterval [-]")

    # -------------------------
    # Verticale lijnen + labels
    # -------------------------
    for x, lab in [
        (voorland, "Hoogte voorland"),
        (achterland, "Hoogte achterland"),
        (kruin, "Kruinhoogte"),
    ]:
        ax1.axvline(x=x, color="k", linestyle=":", linewidth=1.5, alpha=0.2)
        ax1.text(
            x,
            0.1,
            lab,
            transform=ax1.get_xaxis_transform(),
            rotation=90,
            va="baseline",
            ha="right",
            fontsize=9,
            alpha=0.2,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.7, pad=1.5),
        )

    # =========================
    # Legenda samenvoegen
    # =========================
    lines = [l1, l2, l3, bars]
    labels = ["CDF  F(h)=P(H≤h)", "Fragility  P(falen|h)", "PDF  f_H(h)", "Kansbijdrage per Δh"]

    ax1.legend(lines, labels, loc="upper right", frameon=True)

    # =========================
    # Titel
    # =========================
    traj = str(data.Traject[i])
    vak = str(data.vaknummer[i])
    ax1.set_title(f"Vaknummer {vak}\nKerende hoogte dh = {dh:.2f} m\nTotale faalkans Pf = {Faalkans:.2e}")
    ax2.set_yscale("log")
    plt.tight_layout()
    plt.show()

    print(f"De faalkans bedraagt Pf = {Faalkans:.6e}")
    data.iloc[i, -1] = Faalkans
    out_base = r"P:\PR\5536.10\Uitvoering\04 - Fase 1\04 - Analyse\02_geometrischetoets"
    os.makedirs(out_base, exist_ok=True)

    fig_path = os.path.join(out_base, f"{traj}_{vak}.png")

    plt.tight_layout()
    fig.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    plt.close()

In [ ]:
plt.figure(figsize=(8, 4))
plt.semilogy(data.iloc[:, -1], marker="o")
plt.xlabel("Locatie index")
plt.ylabel("Faalkans Pf")
plt.title("Faalkans per locatie")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
data.head()